# 14_Deepen_Winning_Model_Nitrogen_ML_Transfer
## Materia Arche V3.2
### Push the ExtraTrees winner from NB13, explore feature engineering, nitrogen status

NB13 found ExtraTrees (tau-b 0.2675, CV 0.2871) beats the RF baseline (0.2490).
This notebook:
1. Confirms the winner with deeper hyperparameter search
2. Tests feature engineering (interactions, log transforms)
3. Compares ensemble stacking
4. Honest nitrogen transfer assessment

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import (ExtraTreesRegressor, RandomForestRegressor,
                               GradientBoostingRegressor)
from sklearn.model_selection import (RandomizedSearchCV, cross_val_score,
                                      train_test_split, KFold)
from sklearn.metrics import make_scorer, mean_absolute_error
from sklearn.preprocessing import PolynomialFeatures
from scipy.stats import kendalltau
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded")

Libraries loaded


## 1. Reproduce NB13 winner and establish baseline

In [2]:
# Load data — same setup as all previous notebooks
df = pd.read_csv("perovskite_stability_clean.csv")
print(f"Loaded {len(df)} devices")

ml_features = ['Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
               'MA', 'FA', 'Cs',
               'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
               'Perovskite_thickness', 'Cell_area_measured', 'JV_default_Voc',
               'JV_default_Jsc', 'JV_default_FF']

X = df[ml_features].fillna(0)
y = np.log1p(df['Stability_PCE_T80'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def kendall_scorer(y_true, y_pred):
    tau, _ = kendalltau(y_true, y_pred)
    return tau if not np.isnan(tau) else 0.0

scorer = make_scorer(kendall_scorer, greater_is_better=True)

# Reproduce RF baseline (NB02)
rf_baseline = RandomForestRegressor(n_estimators=200, random_state=42)
rf_baseline.fit(X_train, y_train)
pred_rf = rf_baseline.predict(X_test)
tau_rf, _ = kendalltau(y_test, pred_rf)
mae_rf = mean_absolute_error(y_test, pred_rf)

# Reproduce ExtraTrees winner (NB13)
et_nb13 = ExtraTreesRegressor(n_estimators=500, max_features='log2', random_state=42)
et_nb13.fit(X_train, y_train)
pred_et = et_nb13.predict(X_test)
tau_et, _ = kendalltau(y_test, pred_et)
mae_et = mean_absolute_error(y_test, pred_et)

print(f"\nReproduced baselines:")
print(f"  RF (NB02):         tau-b = {tau_rf:.4f}, MAE = {mae_rf:.4f}")
print(f"  ExtraTrees (NB13): tau-b = {tau_et:.4f}, MAE = {mae_et:.4f}")
print(f"  ET lift over RF:   {tau_et - tau_rf:+.4f}")

Loaded 1543 devices



Reproduced baselines:
  RF (NB02):         tau-b = 0.2490, MAE = 1.3053
  ExtraTrees (NB13): tau-b = 0.2675, MAE = 1.2570
  ET lift over RF:   +0.0186


## 2. Deeper ExtraTrees hyperparameter search

In [3]:
# Wider search around the NB13 winner
param_grid_deep = {
    'n_estimators': [300, 500, 700, 1000],
    'max_depth': [None, 20, 30, 50],
    'min_samples_split': [2, 3, 5],
    'min_samples_leaf': [1, 2, 3],
    'max_features': ['sqrt', 'log2', 0.5, 0.7, None],
    'bootstrap': [True, False]
}

print("Deeper ExtraTrees search (50 configs, 5-fold CV)...")
search_deep = RandomizedSearchCV(
    ExtraTreesRegressor(random_state=42),
    param_grid_deep, n_iter=50, cv=5, scoring=scorer,
    random_state=42, n_jobs=-1
)
search_deep.fit(X_train, y_train)

pred_deep = search_deep.predict(X_test)
tau_deep, _ = kendalltau(y_test, pred_deep)
mae_deep = mean_absolute_error(y_test, pred_deep)

print(f"\nDeeper ET search:")
print(f"  Best CV tau-b: {search_deep.best_score_:.4f}")
print(f"  Test tau-b:    {tau_deep:.4f} (lift vs NB13 ET: {tau_deep - tau_et:+.4f})")
print(f"  Test MAE:      {mae_deep:.4f}")
print(f"  Best params:   {search_deep.best_params_}")

Deeper ExtraTrees search (50 configs, 5-fold CV)...



Deeper ET search:
  Best CV tau-b: 0.2836
  Test tau-b:    0.2706 (lift vs NB13 ET: +0.0031)
  Test MAE:      1.2531
  Best params:   {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None, 'bootstrap': False}


## 3. Feature engineering — interactions and transforms

In [4]:
# Test whether engineered features help ExtraTrees
# Tree models can learn interactions natively, but explicit features
# sometimes help when data is limited

# Feature set A: Add composition ratios
X_eng = X.copy()
X_eng['Pb_I_ratio'] = X['Pb'] / (X['I'] + 1e-8)
X_eng['MA_FA_ratio'] = X['MA'] / (X['FA'] + 1e-8)
X_eng['halide_total'] = X['I'] + X['Br'] + X['Cl']
X_eng['cation_total'] = X['MA'] + X['FA'] + X['Cs']
X_eng['Voc_x_Jsc'] = X['JV_default_Voc'] * X['JV_default_Jsc']
X_eng['Voc_x_FF'] = X['JV_default_Voc'] * X['JV_default_FF']
X_eng['log_thickness'] = np.log1p(X['Perovskite_thickness'])
X_eng['log_anneal_time'] = np.log1p(X['first_Prvskt_thermal_annealing_time'])

eng_features = list(X_eng.columns)
print(f"Engineered features: {len(eng_features)} (was {len(ml_features)})")

X_eng_train, X_eng_test = X_eng.iloc[X_train.index], X_eng.iloc[X_test.index]

# Test with NB13 winner params
et_eng = ExtraTreesRegressor(n_estimators=500, max_features='log2', random_state=42)
et_eng.fit(X_eng_train, y_train)
pred_eng = et_eng.predict(X_eng_test)
tau_eng, _ = kendalltau(y_test, pred_eng)
mae_eng = mean_absolute_error(y_test, pred_eng)

print(f"\nExtraTrees + engineered features:")
print(f"  Test tau-b: {tau_eng:.4f} (lift vs plain ET: {tau_eng - tau_et:+.4f})")
print(f"  Test MAE:   {mae_eng:.4f}")

# Test with deeper search params too
et_eng_deep = search_deep.best_estimator_.__class__(
    random_state=42, **search_deep.best_params_
)
et_eng_deep.fit(X_eng_train, y_train)
pred_eng_deep = et_eng_deep.predict(X_eng_test)
tau_eng_deep, _ = kendalltau(y_test, pred_eng_deep)
mae_eng_deep = mean_absolute_error(y_test, pred_eng_deep)

print(f"\nDeep ET + engineered features:")
print(f"  Test tau-b: {tau_eng_deep:.4f} (lift vs plain deep ET: {tau_eng_deep - tau_deep:+.4f})")
print(f"  Test MAE:   {mae_eng_deep:.4f}")

if tau_eng > tau_et or tau_eng_deep > tau_deep:
    print("\nEngineered features helped.")
else:
    print("\nEngineered features did not help — trees already capture these patterns.")

Engineered features: 24 (was 16)



ExtraTrees + engineered features:
  Test tau-b: 0.2519 (lift vs plain ET: -0.0156)
  Test MAE:   1.2821



Deep ET + engineered features:
  Test tau-b: 0.2542 (lift vs plain deep ET: -0.0164)
  Test MAE:   1.2808

Engineered features did not help — trees already capture these patterns.


## 4. Ensemble stacking (ExtraTrees + GBR)

In [5]:
# Simple averaging ensemble: ExtraTrees + GradientBoosting
# GBR was second-best family in NB13 (different inductive bias)

gbr = GradientBoostingRegressor(
    n_estimators=200, max_depth=10, learning_rate=0.01,
    subsample=0.8, min_samples_split=2, random_state=42
)
gbr.fit(X_train, y_train)
pred_gbr = gbr.predict(X_test)
tau_gbr, _ = kendalltau(y_test, pred_gbr)

# Average ensemble
pred_ensemble = (pred_deep + pred_gbr) / 2
tau_ensemble, _ = kendalltau(y_test, pred_ensemble)
mae_ensemble = mean_absolute_error(y_test, pred_ensemble)

# Weighted ensemble (optimize weight on test — note: optimistic estimate)
best_w, best_tau_w = 0.5, tau_ensemble
for w in np.arange(0.1, 1.0, 0.05):
    pred_w = w * pred_deep + (1 - w) * pred_gbr
    tau_w, _ = kendalltau(y_test, pred_w)
    if tau_w > best_tau_w:
        best_w, best_tau_w = w, tau_w

pred_weighted = best_w * pred_deep + (1 - best_w) * pred_gbr
tau_weighted, _ = kendalltau(y_test, pred_weighted)
mae_weighted = mean_absolute_error(y_test, pred_weighted)

print("Ensemble results:")
print(f"  GBR alone:          tau-b = {tau_gbr:.4f}")
print(f"  Deep ET alone:      tau-b = {tau_deep:.4f}")
print(f"  Simple average:     tau-b = {tau_ensemble:.4f} (lift vs ET: {tau_ensemble - tau_deep:+.4f})")
print(f"  Weighted ({best_w:.2f}/{1-best_w:.2f}):  tau-b = {tau_weighted:.4f} (lift vs ET: {tau_weighted - tau_deep:+.4f})")
print(f"  Note: weighted ensemble weight was optimized on test set — optimistic.")

Ensemble results:
  GBR alone:          tau-b = 0.2466
  Deep ET alone:      tau-b = 0.2706
  Simple average:     tau-b = 0.2715 (lift vs ET: +0.0009)
  Weighted (0.80/0.20):  tau-b = 0.2734 (lift vs ET: +0.0028)
  Note: weighted ensemble weight was optimized on test set — optimistic.


## 5. Final leaderboard and CV comparison

In [6]:
# Collect all variants and CV the best ones
variants = {
    "RF baseline (NB02)": (tau_rf, mae_rf),
    "ET NB13 winner": (tau_et, mae_et),
    "ET deeper search": (tau_deep, mae_deep),
    "ET + eng features": (tau_eng, mae_eng),
    "ET deep + eng features": (tau_eng_deep, mae_eng_deep),
    "Ensemble (ET+GBR avg)": (tau_ensemble, mae_ensemble),
    "Ensemble (ET+GBR weighted)": (tau_weighted, mae_weighted),
}

leaderboard = pd.DataFrame([
    {"Variant": k, "Test_Tau_b": round(v[0], 4), "Test_MAE": round(v[1], 4),
     "Lift_vs_RF": round(v[0] - tau_rf, 4)}
    for k, v in variants.items()
]).sort_values("Test_Tau_b", ascending=False)

print("=" * 70)
print("NB14 DEEPENING LEADERBOARD")
print("=" * 70)
print(leaderboard.to_string(index=False))

# CV the top variant (excluding weighted ensemble which was test-optimized)
# Pick the best non-ensemble or simple-ensemble variant
best_variant = leaderboard.iloc[0]['Variant']
print(f"\nBest variant: {best_variant}")

# Cross-validate the deep ET (the most trustworthy improvement)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_taus = []
for train_idx, val_idx in kf.split(X):
    et_cv = ExtraTreesRegressor(random_state=42, **search_deep.best_params_)
    et_cv.fit(X.iloc[train_idx], y.iloc[train_idx])
    pred_cv = et_cv.predict(X.iloc[val_idx])
    tau_cv, _ = kendalltau(y.iloc[val_idx], pred_cv)
    cv_taus.append(tau_cv)

print(f"\nDeep ET 5-fold CV: {np.mean(cv_taus):.4f} +/- {np.std(cv_taus):.4f}")
print(f"(vs RF baseline CV from NB13: 0.2569 +/- 0.0125)")

leaderboard.to_csv("NB14_Deepening_Leaderboard.csv", index=False)
print("\nNB14_Deepening_Leaderboard.csv exported")

NB14 DEEPENING LEADERBOARD
                   Variant  Test_Tau_b  Test_MAE  Lift_vs_RF
Ensemble (ET+GBR weighted)      0.2734    1.2542      0.0244
     Ensemble (ET+GBR avg)      0.2715    1.2675      0.0226
          ET deeper search      0.2706    1.2531      0.0216
            ET NB13 winner      0.2675    1.2570      0.0186
    ET deep + eng features      0.2542    1.2808      0.0052
         ET + eng features      0.2519    1.2821      0.0030
        RF baseline (NB02)      0.2490    1.3053      0.0000

Best variant: Ensemble (ET+GBR weighted)



Deep ET 5-fold CV: 0.2868 +/- 0.0123
(vs RF baseline CV from NB13: 0.2569 +/- 0.0125)

NB14_Deepening_Leaderboard.csv exported


## 6. Nitrogen ML transfer — honest assessment

In [7]:
print("=" * 65)
print("NITROGEN ML TRANSFER — HONEST ASSESSMENT")
print("=" * 65)
print("")
print("What transfers directly from perovskite work:")
print("  - Pipeline architecture (data -> features -> model -> candidates)")
print("  - Statistical validation (Kendall tau-b, CV, bootstrap)")
print("  - Evidence packaging and reproducibility framework")
print("  - Model selection methodology (NB13-14 sweep approach)")
print("  - ExtraTrees as strong default for materials regression")
print("")
print("What does NOT transfer:")
print("  - Features: perovskite composition (Pb/Sn/I/Br/MA/FA/Cs) is")
print("    domain-specific. Nitrogen fixation needs different descriptors")
print("    (d-band center, binding energies, coordination numbers)")
print("  - Target: T80 stability is perovskite-specific.")
print("    Nitrogen needs N2 binding energy or turnover frequency")
print("  - Data: No equivalent open database for N2 fixation catalysts")
print("")
print("What nitrogen needs before ML can start:")
print("  1. Catalyst composition database (scrape literature or compute)")
print("  2. DFT-computed binding energies as ground truth")
print("  3. Domain-specific features for N2 fixation catalysis")
print("  4. Estimated: 3-6 months of dedicated work")
print("")
print("NOTE: Feeding random numbers into the perovskite model and calling")
print("it 'nitrogen transfer' would be scientifically dishonest.")
print("The methodology transfers. The model and data do not.")
print("")
print("STATUS: ON HOLD until perovskite Phase 2 results are in.")

NITROGEN ML TRANSFER — HONEST ASSESSMENT

What transfers directly from perovskite work:
  - Pipeline architecture (data -> features -> model -> candidates)
  - Statistical validation (Kendall tau-b, CV, bootstrap)
  - Evidence packaging and reproducibility framework
  - Model selection methodology (NB13-14 sweep approach)
  - ExtraTrees as strong default for materials regression

What does NOT transfer:
  - Features: perovskite composition (Pb/Sn/I/Br/MA/FA/Cs) is
    domain-specific. Nitrogen fixation needs different descriptors
    (d-band center, binding energies, coordination numbers)
  - Target: T80 stability is perovskite-specific.
    Nitrogen needs N2 binding energy or turnover frequency
  - Data: No equivalent open database for N2 fixation catalysts

What nitrogen needs before ML can start:
  1. Catalyst composition database (scrape literature or compute)
  2. DFT-computed binding energies as ground truth
  3. Domain-specific features for N2 fixation catalysis
  4. Estimated: 

## 7. Summary

In [8]:
print("=" * 65)
print("SUMMARY — NOTEBOOK 14")
print("=" * 65)
print("")
print("Starting point: ExtraTrees (NB13) tau-b 0.2675, CV 0.2871")
print("")
print("Experiments:")
print(f"  1. Deeper ET search (50 configs):     tau-b = {tau_deep:.4f}")
print(f"  2. Feature engineering (8 new feats):  tau-b = {tau_eng:.4f}")
print(f"  3. Deep ET + eng features:             tau-b = {tau_eng_deep:.4f}")
print(f"  4. ET+GBR ensemble (average):          tau-b = {tau_ensemble:.4f}")
print(f"  5. ET+GBR ensemble (weighted):         tau-b = {tau_weighted:.4f} (test-optimized)")
print("")
print(f"Best trustworthy model: Deep ET")
print(f"  Test tau-b: {tau_deep:.4f}")
print(f"  CV tau-b:   {np.mean(cv_taus):.4f} +/- {np.std(cv_taus):.4f}")
print(f"  Lift vs RF baseline: {tau_deep - tau_rf:+.4f}")
print("")
print("Conclusion:")
print("  ExtraTrees with deeper tuning is the best single model.")
print("  Feature engineering provides marginal/no improvement")
print("  (trees already learn these interactions).")
print("  Ensemble averaging may help but needs CV validation.")
print("")
print("Nitrogen: Methodology proven. Data pipeline not yet built.")
print("")
print("Phase 2 status: ACTIVE")
print("Best model for Phase 2 candidates: ExtraTrees")
print("Notebooks: 14 (01-14)")

SUMMARY — NOTEBOOK 14

Starting point: ExtraTrees (NB13) tau-b 0.2675, CV 0.2871

Experiments:
  1. Deeper ET search (50 configs):     tau-b = 0.2706
  2. Feature engineering (8 new feats):  tau-b = 0.2519
  3. Deep ET + eng features:             tau-b = 0.2542
  4. ET+GBR ensemble (average):          tau-b = 0.2715
  5. ET+GBR ensemble (weighted):         tau-b = 0.2734 (test-optimized)

Best trustworthy model: Deep ET
  Test tau-b: 0.2706
  CV tau-b:   0.2868 +/- 0.0123
  Lift vs RF baseline: +0.0216

Conclusion:
  ExtraTrees with deeper tuning is the best single model.
  Feature engineering provides marginal/no improvement
  (trees already learn these interactions).
  Ensemble averaging may help but needs CV validation.

Nitrogen: Methodology proven. Data pipeline not yet built.

Phase 2 status: ACTIVE
Best model for Phase 2 candidates: ExtraTrees
Notebooks: 14 (01-14)
